# Predicción sobre test

Carga `modelo_final.joblib` y los 5 BETO de `modelos_beto/` (generados por `train.ipynb`), promedia 50/50 las probabilidades de los dos modelos y genera `submission/submission.csv`. Contiene **solo** el código necesario para replicar la submission.

In [1]:
CON_BETO = True  # False reproduce la submission de respaldo: modelo sin BETO, carga modelo_final_p7c_duracion.joblib y escribe submission/submission.csv

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from texto_beto import predecir_beto_promedio
from features import aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos

/Users/nicolasscravaglieri/Downloads/kaggle-ap-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga del test y de los modelos

Al test se le aplican **las mismas cuatro funciones determinísticas** que al train en `train.ipynb`. Todo lo que aprende de los datos ya está ajustado, con el train, adentro de los dos modelos de `modelo_final.joblib`.

La probabilidad de BETO del test (`prob_beto_new`) es el **promedio de los 5 BETO** que entrenó `train.ipynb`. Ninguno vio el test, así que no hace falta el *cross-fitting* que sí se usa en el train.

In [2]:
test = pd.read_json("data/test_data.jsonlines", lines=True)
test = aplanar_anidadas(test)
test = agregar_features_contenido(test)
test = agregar_categoricas_crudas(test)
test = agregar_combos(test)
if CON_BETO:
    test["prob_beto_new"] = predecir_beto_promedio("modelos_beto", test["title"], test["warranty_texto"])

archivo_modelo = "modelo_final.joblib" if CON_BETO else "modelo_final_p7c_duracion.joblib"
modelos = joblib.load(archivo_modelo)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9111.94it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8831.59it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9045.46it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9335.43it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9643.51it/s]

## 2. Promedio 50/50 y submission

P(`new`) = 0,5 · P(`new`) de CatBoost + 0,5 · P(`new`) del HistGB. La clase es `new` si ese promedio es ≥ 0,5.

Formato confirmado por Kaggle: CSV con columnas `ID` y `condition`.
- `ID` = número de fila **empezando en 1** (1 a 30.000), en el orden de `test_data.jsonlines`.
- `condition` = `"new"` o `"used"`.

In [3]:
prob_catboost = modelos["catboost"].predict_proba(test)[:, 1]
prob_histgb = modelos["histgb"].predict_proba(test)[:, 1]
prob_new = 0.5 * prob_catboost + 0.5 * prob_histgb

submission = pd.DataFrame({
    "ID": np.arange(1, len(test) + 1),
    "condition": np.where(prob_new >= 0.5, "new", "used"),
})

Path("submission").mkdir(exist_ok=True)
submission.to_csv("submission/submission.csv", index=False)

## 3. Chequeo de formato

Sobre el archivo leído desde el disco: 30.000 filas, columnas `ID` y `condition`, `ID` de 1 a 30.000 en orden, y `condition` solo `"new"`/`"used"`.

In [4]:
leida = pd.read_csv("submission/submission.csv")

assert leida.shape == (30000, 2)
assert list(leida.columns) == ["ID", "condition"]
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()
print("Formato OK")

Formato OK
